# 单约束校准 v2（DA / CoT，只靠 system prompt 区分）

相对 v1 的改动（依据 4B 的原始输出诊断）：

| 问题 | v2 处理 |
|---|---|
| 模型写 `STATUS = DONE` / `PARITY= ODD`（等号旁有空格）被判错 | keyword 行判定改为容忍等号两侧空格 |
| `PARITY`、`ARGMIN` 行排在 `####ANSWER` 之后 | 每条 keyword 约束明确写"放在最后一行之前，不得放在其后" |
| K1 在 d=1 把起点也算一步 | 改成"编号运算列表里最大的编号" |
| K3 的"整数部分"对负数有歧义 | 作废，换成 `COUNT_ADD`（加法运算的个数） |
| L2 的行数上限在 DA/CoT 下不同，违反"cons 完全相同" | 两种模式共用 `3d+4` |
| DA 在 d≤10 时偷偷写推理（违约率约 60%） | **加强 DA preamble**，先用 `da_recheck` 阶段对比新旧版本再决定是否采用 |
| 通过率全是天花板（F2/F3/V1/V3/L1） | 删除；新增会和 CoT trace 竞争的约束 W1/W2/W3 |
| `cons_ok` 不含 `base_ok` | 并入，`cons_only` 另列 |
| 入选规则要求两种模式都在 30%–85%，会排除"CoT 成本型"约束 | 改为：最低通过率 ≥ 30%、至少一种模式 ≤ 85%、`base_ok ≥ 90%`、DA 违约 ≤ 20%（**事后修订**，写进 record.md） |

只有 DA preamble 的措辞会改动 AGENTS 4.1，且要等 `da_recheck` 的结果；CoT preamble 保持不变。

**使用顺序**：配置 -> 题目 -> 约束 -> prompt -> 加载 vLLM -> 生成器 -> `da_recheck` -> 完整校准 -> 汇总 -> k 轴。
结果写入 `calib_v2_results_tier{TIER}.jsonl`，不会覆盖 v1 的结果。

In [1]:
# Colab / Kaggle 专用：安装 vLLM（本地 WSL 环境请跳过）
import sys, os, subprocess
if "google.colab" in sys.modules or os.path.exists("/kaggle"):
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "vllm", "pandas"], check=True)
    print("安装完成。如果提示 restart，请重启会话后从下一格开始。")

安装完成。如果提示 restart，请重启会话后从下一格开始。


In [2]:
import os, sys, json, re, time, random, gc, math

# 必须在 import vllm 之前设置（AGENTS 第 3 节）
os.environ["VLLM_USE_FLASHINFER_SAMPLER"] = "0"

import numpy as np
import pandas as pd
import torch

IN_COLAB = "google.colab" in sys.modules

# ---------------- 配置 ----------------
TIER = "B"   # "A": Qwen3-1.7B；"B": Qwen3-4B-Instruct-2507
MODEL_SPECS = {
    "A": ("Qwen/Qwen3-1.7B", "models/Qwen3-1.7B", True),
    "B": ("Qwen/Qwen3-4B-Instruct-2507", "models/Qwen3-4B-Instruct-2507", False),
}
PROMPT_VERSION = "v3b-da-cot-calib"
MAX_TOKENS = {"DA": 512, "DA_old": 512, "CoT": 2048}   # v1 的 CoT p95 约 930，d=20 均值约 780，2048 留足余量
TEMPERATURE, TOP_P = 0.6, 0.95

D_LIST = [1, 5, 10, 15, 20]
N_PER_D = 20
SEED = 20261105

MOUNT_DRIVE = True    # Colab 上强烈建议 True，防止额度用完/断线丢结果
OUT_DIR = "."
if IN_COLAB and MOUNT_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    OUT_DIR = "/content/drive/MyDrive/cisc7404_calib"
    os.makedirs(OUT_DIR, exist_ok=True)
elif os.path.exists("/kaggle"):
    OUT_DIR = "/kaggle/working"
OUT_PATH = f"{OUT_DIR}/calib_v2_results_tier{TIER}.jsonl"
print("TIER", TIER, MODEL_SPECS[TIER][0], "| out:", OUT_PATH)
print("CUDA:", torch.cuda.is_available(), torch.cuda.get_device_name(0) if torch.cuda.is_available() else "-")

Mounted at /content/drive
TIER B Qwen/Qwen3-4B-Instruct-2507 | out: /content/drive/MyDrive/cisc7404_calib/calib_v2_results_tierB.jsonl
CUDA: True Tesla T4


## 1. 校准题目

与 v1 完全相同的题目（同一 seed）；新增字段 `n_add`（加法运算个数），供 `COUNT_ADD` 判分。题干与答案不变。

In [3]:
def fmt_tenths(t):
    sign = "-" if t < 0 else ""
    return f"{sign}{abs(t)//10}.{abs(t)%10}"

def make_question(d, i):
    rng = random.Random(f"{SEED}-{d}-{i}")
    x0 = rng.randint(-50, 150)
    vals, steps = [x0], []
    for _ in range(d):
        delta = rng.randint(1, 99)
        sign = rng.choice([1, -1])
        vals.append(vals[-1] + sign * delta)
        steps.append(("add" if sign > 0 else "subtract", delta))
    mn = min(vals)
    ops = "\n".join(f"{j+1}. {w} {fmt_tenths(dl)}" for j, (w, dl) in enumerate(steps))
    q = (f"A number starts at {fmt_tenths(x0)}. Apply these operations in order:\n{ops}\n"
         "Over the whole process, including the starting value, what is the smallest value the number takes?")
    return {"qid": f"d{d}_i{i}", "d": d, "i": i, "W": d, "q": q,
            "ans_tenths": mn, "ans": fmt_tenths(mn),
            "argmin": [j for j, v in enumerate(vals) if v == mn],
            "n_add": sum(1 for w, _ in steps if w == "add")}

QUESTIONS = [make_question(d, i) for d in D_LIST for i in range(N_PER_D)]
print(len(QUESTIONS), "题；样例 n_add =", QUESTIONS[12]["n_add"])

100 题；样例 n_add = 1


## 2. 约束候选池 v2

- 全部 keyword 行（`STATUS / STEPS / ARGMIN / COUNT_ADD`）判定**容忍等号两侧空格**，因为 prompt 没规定空格。
- 需要额外输出一行的约束，统一写明"放在最后一行之前"。
- `L3` 的行数上限对 DA / CoT **相同**：`3d+4`（v1 的 CoT 行数 p50 约为 3d–3.5d）。
- 新增的竞争型约束：`W1` 禁用 add/subtract（CoT 会照抄题干）、`W2` 明确禁逗号（含数字列表）、`W3` 除 keyword 行外不得出现等号（CoT 逐步写 `a + b = c`）。
- `K2` 与推理强耦合，单独归为内容类，汇总时单独看。

In [4]:
MARK = "####ANSWER="
KEYS = ("####ANSWER", "STATUS", "STEPS", "ARGMIN", "COUNT_ADD")
KEYLINE = re.compile(r"^(" + "|".join(re.escape(k) for k in KEYS) + r")\s*=")

def lines_of(t):
    return [l.strip() for l in t.strip().splitlines() if l.strip()]

def marker_lines(t):
    return [l for l in lines_of(t) if l.startswith(MARK)]

def final_value(t):
    ml = [l for l in t.splitlines() if MARK in l]
    return ml[-1].split(MARK)[-1].strip() if ml else None

def to_float(s):
    try:
        return float(s.strip().strip("[]"))
    except Exception:
        return None

def kv_values(t, key):
    # 容忍 KEY = VALUE / KEY=VALUE / KEY= VALUE
    out = []
    for l in lines_of(t):
        m = re.fullmatch(key + r"\s*=\s*(\S+)", l)
        if m:
            out.append(m.group(1))
    return out

def chk_F2(t, r):
    v = final_value(t);  return v is not None and re.fullmatch(r"\[?-?\d+\.\d{2}\]?", v) is not None
def chk_F4(t, r):
    return any(re.fullmatch(r"STATUS\s*=\s*DONE", l) for l in lines_of(t)[:-1])
def chk_K1(t, r):
    return any(v.isdigit() and int(v) == r["d"] for v in kv_values(t, "STEPS"))
def chk_K2(t, r):
    return any(v.isdigit() and int(v) in r["argmin"] for v in kv_values(t, "ARGMIN"))
def chk_K4(t, r):
    return any(v.isdigit() and int(v) == r["n_add"] for v in kv_values(t, "COUNT_ADD"))
def chk_W1(t, r):
    return re.search(r"\b(add|adds|added|adding|subtract|subtracts|subtracted|subtracting)\b", t, re.I) is None
def chk_W2(t, r):
    return "," not in t
def chk_W3(t, r):
    return all("=" not in l for l in lines_of(t) if not KEYLINE.match(l))
def chk_L3(t, r):
    return len(lines_of(t)) <= r["lmax"]

BEFORE = " Place this line before the final line, never after it."
CONS = {
    "F0": dict(scope="base", text=None, check=lambda t, r: True),
    "F2": dict(scope="final", check=chk_F2,
               text="The value after the equals sign must have exactly two digits after the decimal point."),
    "F4": dict(scope="final", check=chk_F4,
               text="Include one line that consists only of the word STATUS, an equals sign, and the word DONE." + BEFORE),
    "K1": dict(scope="final", check=chk_K1,
               text="Include one line that starts with STEPS= followed by the highest number in the numbered list of operations." + BEFORE),
    "K2": dict(scope="final", check=chk_K2,
               text="Include one line that starts with ARGMIN= followed by the index at which the smallest value occurs; the starting value has index 0 and the value after the first operation has index 1." + BEFORE),
    "K4": dict(scope="final", check=chk_K4,
               text="Include one line that starts with COUNT_ADD= followed by how many of the listed operations are additions." + BEFORE),
    "W1": dict(scope="full", check=chk_W1,
               text="Never use the words 'add' or 'subtract' (in any form) anywhere in the response."),
    "W2": dict(scope="full", check=chk_W2,
               text="Never use a comma character anywhere in the response, including inside lists of numbers and after words such as 'Now'."),
    "W3": dict(scope="full", check=chk_W3,
               text="Never use the equals sign anywhere in the response, except in the keyword lines that these rules require."),
    "L3": dict(scope="full", check=chk_L3,
               text="The whole response may contain at most {N} non-empty lines."),
}
CANDIDATES = [c for c in CONS if c != "F0"]
print(CANDIDATES)

['F2', 'F4', 'K1', 'K2', 'K4', 'W1', 'W2', 'W3', 'L3']


## 3. Prompt 拼装与判分

- `CoT` preamble 与 v1 完全相同。
- `DA_old` 是 v1 的 DA preamble，`DA` 是加强版（要求除规则行外不得有任何文字）。两者只在 `da_recheck` 阶段对比；**正式校准只用 `DA`**。
- `cons` 对所有模式完全相同，`lmax = 3d+4` 与模式无关。

In [5]:
CoT_PREAMBLE = ("Solve the problem step by step. Before the final answer, explicitly write the "
                "value after each operation, one per line. Then apply the rules below to the "
                "final answer.\n")
PREAMBLE = {
    "CoT": CoT_PREAMBLE,
    "DA_old": ("Solve the problem silently. Do not show intermediate steps and do not explain "
               "your reasoning. Output only what the rules below require.\n"),
    "DA": ("Solve the problem silently. Your response must contain nothing except the lines required by "
           "the rules below: no calculations, no intermediate values, no explanation and no comments about "
           "the problem. Any additional text counts as an error.\n"),
}
BASE_RULE = ("The last line of your response must begin with ####ANSWER= immediately followed by "
             "the smallest value as a plain decimal number (unless another rule says otherwise), "
             "with nothing after it.")

def lmax_for(d):          # 与模式无关：DA / CoT 的 cons 必须完全相同
    return 3 * d + 4

def build_cons(ids, rec):
    rules = [BASE_RULE] + [CONS[c]["text"].format(N=rec["lmax"]) for c in ids if CONS[c]["text"]]
    return "Rules:\n" + "\n".join(f"R{j+1}. {r}" for j, r in enumerate(rules))

def build_messages(mode, ids, q, rec):
    return [{"role": "system", "content": PREAMBLE[mode] + build_cons(ids, rec)},
            {"role": "user", "content": q["q"]}]

def make_jobs(stage, questions, modes, cons_lists):
    jobs = []
    for mode in modes:
        for ids in cons_lists:
            for q in questions:
                cid = "+".join(ids)
                rec = {"id": f"{PROMPT_VERSION}|{TIER}|{stage}|{mode}|{cid}|{q['qid']}",
                       "stage": stage, "mode": mode, "cons_id": cid, "cons_ids": list(ids),
                       "k": len([c for c in ids if c != "F0"]),
                       "qid": q["qid"], "d": q["d"], "W": q["W"],
                       "ans_tenths": q["ans_tenths"], "argmin": q["argmin"], "n_add": q["n_add"],
                       "lmax": lmax_for(q["d"]), "prompt_version": PROMPT_VERSION,
                       "tier": TIER, "max_tokens": MAX_TOKENS[mode]}
                rec["messages"] = build_messages(mode, ids, q, rec)
                jobs.append(rec)
    return jobs

def score(rec):
    t, mode, d = rec["text"], rec["mode"], rec["d"]
    is_da = mode.startswith("DA")
    lines = lines_of(t)
    v = final_value(t)
    base_ok = bool(lines) and lines[-1].startswith(MARK) and v is not None and \
        re.fullmatch(r"\[?-?\d+(\.\d+)?\]?", v) is not None
    vf = to_float(v) if v is not None else None
    answer_ok = vf is not None and abs(vf - rec["ans_tenths"] / 10) < 0.005
    per = {c: bool(CONS[c]["check"](t, rec)) for c in rec["cons_ids"]}
    cons_only = all(per.values())
    other = [l for l in lines if not KEYLINE.match(l)]
    da_violation = is_da and len(other) > 0
    trace = [l for l in other if re.search(r"\d", l)]
    cot_violation = (not is_da) and len(trace) < max(1, d // 2)
    copy_flag = len(marker_lines(t)) >= 3 or re.search(r"<(number|answer|value)>", t) is not None
    return {"base_ok": base_ok, "answer_ok": answer_ok, "cons_only": cons_only,
            "cons_ok": base_ok and cons_only, "per_cons": per,
            "viol": da_violation if is_da else cot_violation,
            "da_violation": da_violation, "cot_violation": cot_violation,
            "copy_flag": copy_flag, "n_lines": len(lines), "n_commas": t.count(",")}

# ---- 判定函数自测（不需要 GPU）：用 v1 在 4B 上看到的真实输出片段 ----
_r = {"d": 5, "lmax": 19, "argmin": [1], "n_add": 3, "ans_tenths": -46}
_f4 = "Start: 0.7\n\n1. Subtract 5.3:  \n0.7 - 5.3 = -4.6\n\nSTATUS = DONE  \n####ANSWER= -4.6"
assert chk_F4(_f4, _r), "F4 应容忍空格"
assert not chk_F4("####ANSWER=1.0\nSTATUS=DONE", _r), "STATUS 在最后一行之后不应通过"
assert kv_values("COUNT_ADD= 3\n####ANSWER=1.0", "COUNT_ADD") == ["3"] and chk_K4("COUNT_ADD= 3\n####ANSWER=1.0", _r)
assert not chk_W1("1. Subtract 5.3", _r) and chk_W1("1. minus 5.3 gives -4.6", _r)
assert not chk_W3(_f4, _r) and chk_W3("1. 0.7 minus 5.3 gives -4.6\nSTATUS=DONE\n####ANSWER=-4.6", _r)
assert not chk_W2("Now, list", _r) and chk_W2("Now list", _r)
assert chk_L3("a\nb\n####ANSWER=1", _r) and not chk_L3("\n".join(["x"] * 25), _r)
assert chk_K1("STEPS = 5\n####ANSWER=1", _r) and not chk_K1("STEPS=2\n####ANSWER=1", _r)
assert PREAMBLE["CoT"] == CoT_PREAMBLE and lmax_for(10) == 34
print("判定函数自测通过")

判定函数自测通过


## 4. 加载 vLLM（按机器自动选参数）

4B 在 Colab T4 上已验证可用：`dtype=half / gpu_memory_utilization=0.90 / max_model_len=8192 / max_num_seqs=32`。
跑通后记得回填 AGENTS 3.3 与 `record.md`。

In [6]:
from vllm import LLM, SamplingParams

hf_id, local_dir, pass_thinking_flag = MODEL_SPECS[TIER]
model_path = local_dir if os.path.isdir(local_dir) else hf_id

cap = torch.cuda.get_device_capability(0)
mem_gib = torch.cuda.get_device_properties(0).total_memory / 2**30
kw = dict(model=model_path, max_model_len=8192)
if mem_gib < 12:
    if TIER == "B":
        raise NotImplementedError("8GB 卡放不下 bf16 的 4B：先按 AGENTS 3.3 选 fp8 / int4 / 换机器")
    kw.update(gpu_memory_utilization=0.80, max_num_seqs=8)
else:
    kw.update(gpu_memory_utilization=0.85 if TIER == "A" else 0.90, max_num_seqs=32)
if cap[0] < 8:
    kw["dtype"] = "half"

print("LLM kwargs:", kw, "| GPU", f"{mem_gib:.1f} GiB", cap)
llm = LLM(**kw)
CHAT_KW = {"chat_template_kwargs": {"enable_thinking": False}} if pass_thinking_flag else {}
SP = {m: SamplingParams(temperature=TEMPERATURE, top_p=TOP_P, max_tokens=MAX_TOKENS[m]) for m in MAX_TOKENS}

W1105 16:01:25.599000 338 torch/_opaque_base.py:6] torch._opaque_base is deprecated, use torch._custom_class_base instead
W1105 16:01:25.606000 338 torch/_library/opaque_object.py:288] register_opaque_type is deprecated, use register_custom_class instead
W1105 16:01:25.609000 338 torch/_library/opaque_object.py:211] typ='value' is deprecated, use typ='constant' instead


LLM kwargs: {'model': 'Qwen/Qwen3-4B-Instruct-2507', 'max_model_len': 8192, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 32, 'dtype': 'half'} | GPU 14.6 GiB (7, 5)
(MainProcess pid=338) INFO 11-05 16:01:53 [startup.py:96] non-default args: {'dtype': 'half', 'max_model_len': 8192, 'gpu_memory_utilization': 0.9, 'max_num_seqs': 32, 'disable_log_stats': True, 'model': 'Qwen/Qwen3-4B-Instruct-2507'}


config.json:   0%|          | 0.00/727 [00:00<?, ?B/s]

(MainProcess pid=338) INFO 11-05 16:02:17 [model.py:744] Resolved architecture: Qwen3ForCausalLM
(MainProcess pid=338) WARNING 11-05 16:02:17 [model.py:2516] Casting torch.bfloat16 to torch.float16.
(MainProcess pid=338) INFO 11-05 16:02:17 [model.py:2170] Using max model len 8192
(MainProcess pid=338) INFO 11-05 16:02:17 [scheduler.py:310] Chunked prefill is enabled with max_num_batched_tokens=8192.


model.safetensors.index.json:   0%|          | 0.00/32.8k [00:00<?, ?B/s]

Parse safetensors files:   0%|          | 0/3 [00:00<?, ?it/s]

(MainProcess pid=338) INFO 11-05 16:02:18 [kernel.py:480] Final IR op priority after setting platform defaults: IrOpPriorityConfig(rms_norm=['native'], fused_add_rms_norm=['native'], gelu_and_mul_sparse=['triton', 'native'])


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

(MainProcess pid=338) WARNING 11-05 16:02:24 [system_utils.py:154] We must use the `spawn` multiprocessing start method. Overriding VLLM_WORKER_MULTIPROC_METHOD to 'spawn'. See https://docs.vllm.ai/en/latest/usage/troubleshooting.html#python-multiprocessing for more information. Reasons: CUDA is initialized
(MainProcess pid=338) INFO 11-05 16:06:48 [hf.py:736] Detected the chat template content format to be 'string'. You can set `--chat-template-content-format` to override this.


## 5. 生成器（追加写 JSONL，断点续跑，记录 token）

按任务里实际出现的模式分组调用（不同模式 `max_tokens` 不同）。

In [7]:
def load_done(path):
    if not os.path.exists(path):
        return set()
    with open(path, encoding="utf-8") as f:
        return {json.loads(l)["id"] for l in f if l.strip()}

def run_jobs(jobs, path=OUT_PATH, batch=64):
    done = load_done(path)
    todo = [j for j in jobs if j["id"] not in done]
    print(f"总计 {len(jobs)}，已完成 {len(jobs)-len(todo)}，待跑 {len(todo)}")
    for mode in dict.fromkeys(j["mode"] for j in todo):
        part = [j for j in todo if j["mode"] == mode]
        for s in range(0, len(part), batch):
            chunk = part[s:s + batch]
            t0 = time.time()
            outs = llm.chat([j["messages"] for j in chunk], SP[mode], use_tqdm=False, **CHAT_KW)
            dt = time.time() - t0
            with open(path, "a", encoding="utf-8") as f:
                for j, o in zip(chunk, outs):
                    c = o.outputs[0]
                    rec = dict(j)
                    rec.update(text=c.text, prompt_tokens=len(o.prompt_token_ids),
                               n_tokens=len(c.token_ids), finish_reason=c.finish_reason,
                               trunc=(c.finish_reason == "length"), batch_sec=round(dt, 2),
                               batch_size=len(chunk))
                    f.write(json.dumps(rec, ensure_ascii=False) + "\n")
            print(f"[{mode}] {s+len(chunk)}/{len(part)}  batch {dt:.1f}s")

def load_results(path=OUT_PATH, stage=None):
    rows = [json.loads(l) for l in open(path, encoding="utf-8") if l.strip()]
    df = pd.DataFrame(rows)
    if stage:
        df = df[df["stage"] == stage].copy()
    sc = [score(r) for r in df.to_dict("records")]
    return pd.concat([df.reset_index(drop=True), pd.DataFrame(sc)], axis=1)

## 6. 第一步：DA preamble 复测（`da_recheck`，约 300 条，几分钟）

用全新的题（`i=60..79`），在 F0 下对比三种：`DA_old`（v1）、`DA`（加强版）、`CoT`（参照）。

**采用规则**：加强版 DA 的违约率（`viol`）≤ 10%，且 DA 不会因此全部输出空白/格式错乱（`base_ok` ≥ 95%），才采用，并同步更新 AGENTS 4.1 与 `record.md`。
否则回来讨论，不要硬套。

In [8]:
RE_Q = [make_question(d, i) for d in D_LIST for i in range(60, 80)]
run_jobs(make_jobs("da_recheck", RE_Q, ["DA_old", "DA", "CoT"], [["F0"]]))

rc = load_results(stage="da_recheck")
print(rc.groupby("mode").agg(answer=("answer_ok", "mean"), base_ok=("base_ok", "mean"), viol=("viol", "mean"),
                             trunc=("trunc", "mean"), tok_mean=("n_tokens", "mean"),
                             tok_p95=("n_tokens", lambda s: s.quantile(0.95))).round(3))
print("\n违约率 viol（按 d）：")
print(rc.pivot_table(index="mode", columns="d", values="viol").round(2))
print("\n答案正确率（按 d）：")
print(rc.pivot_table(index="mode", columns="d", values="answer_ok").round(2))
print("\n平均 token（按 d）：")
print(rc.pivot_table(index="mode", columns="d", values="n_tokens").round(0))
r = rc[(rc["mode"] == "DA") & (rc["d"] == 10)].iloc[0]
print("\n===== DA（加强版）d=10 示例 =====\n" + r["text"])

总计 300，已完成 0，待跑 300
[DA_old] 64/100  batch 37.9s
[DA_old] 100/100  batch 14.9s
[DA] 64/100  batch 9.0s
[DA] 100/100  batch 4.7s
[CoT] 64/100  batch 65.4s
[CoT] 100/100  batch 125.6s
        answer  base_ok  viol  trunc  tok_mean  tok_p95
mode                                                   
CoT       0.96      1.0  0.00    0.0    407.29   798.00
DA        0.30      1.0  0.11    0.0     11.92    37.00
DA_old    0.58      1.0  0.60    0.0     93.39   258.45

违约率 viol（按 d）：
d         1     5     10   15   20
mode                              
CoT     0.00  0.00  0.00  0.0  0.0
DA      0.45  0.05  0.05  0.0  0.0
DA_old  0.90  1.00  1.00  0.1  0.0

答案正确率（按 d）：
d         1     5     10    15    20
mode                                
CoT     1.00  0.90  0.95  0.95  1.00
DA      0.90  0.25  0.20  0.15  0.00
DA_old  0.95  0.95  0.75  0.20  0.05

平均 token（按 d）：
d         1      5      10     15     20
mode                                    
CoT     94.0  190.0  398.0  574.0  780.0
DA      18

## 7. 完整单约束校准

`10 条（含基线）× 2 模式 × 100 题 = 2000 条`。v1 在 4B 上生成约 47 分钟（2400 条），本轮预计相近；DA 违约变少后会更快。
可中断，重跑自动续上。**请先确认第 6 步的结果再跑这一格。**

In [9]:
run_jobs(make_jobs("calib", QUESTIONS, ["DA", "CoT"], [[c] for c in ["F0"] + CANDIDATES]))

总计 2000，已完成 0，待跑 2000
[DA] 64/1000  batch 7.9s
[DA] 128/1000  batch 9.8s
[DA] 192/1000  batch 9.5s
[DA] 256/1000  batch 5.3s
[DA] 320/1000  batch 7.4s
[DA] 384/1000  batch 7.0s
[DA] 448/1000  batch 6.3s
[DA] 512/1000  batch 8.0s
[DA] 576/1000  batch 6.1s
[DA] 640/1000  batch 5.8s
[DA] 704/1000  batch 8.2s
[DA] 768/1000  batch 9.9s
[DA] 832/1000  batch 10.6s
[DA] 896/1000  batch 31.6s
[DA] 960/1000  batch 37.1s
[DA] 1000/1000  batch 53.3s
[CoT] 64/1000  batch 70.5s
[CoT] 128/1000  batch 144.3s
[CoT] 192/1000  batch 142.8s
[CoT] 256/1000  batch 79.3s
[CoT] 320/1000  batch 146.0s
[CoT] 384/1000  batch 173.9s
[CoT] 448/1000  batch 94.3s
[CoT] 512/1000  batch 158.9s
[CoT] 576/1000  batch 92.3s
[CoT] 640/1000  batch 111.3s
[CoT] 704/1000  batch 135.5s
[CoT] 768/1000  batch 64.2s
[CoT] 832/1000  batch 114.7s
[CoT] 896/1000  batch 125.3s
[CoT] 960/1000  batch 65.7s
[CoT] 1000/1000  batch 148.5s


## 8. 汇总与选约束（修订后的规则）

每条约束归为四类：

- **结构问题**：任一模式 `base_ok < 90%`（keyword 行排到最后一行之后等），先修 prompt，不看通过率
- **地板**：任一模式 `cons_pass_stop < 30%`
- **天花板**：两种模式都 `> 85%`，进 k 轴也不会产生梯度
- **可用**：最低通过率 ≥ 30%，且至少一种模式 ≤ 85%

`SELECTED` = 可用 + 截断 < 5% + 复制 ≤ 1% + DA 违约 ≤ 20%。这是看了 v1 数据后的**事后修订**。
`cons_pass_stop` 已包含 `base_ok`；`cons_only` 是只看约束本身。

In [10]:
BASE_MIN, LO, HI, TRUNC_MAX, COPY_MAX, DA_VIOL_MAX = 0.90, 0.30, 0.85, 0.05, 0.01, 0.20
cal = load_results(stage="calib")
cal["scope"] = cal["cons_id"].map(lambda c: CONS[c]["scope"])

g = cal.groupby(["cons_id", "mode"])
summ = g.agg(n=("id", "size"), scope=("scope", "first"), cons_pass=("cons_ok", "mean"), cons_only=("cons_only", "mean"),
             base_ok=("base_ok", "mean"), answer=("answer_ok", "mean"),
             trunc=("trunc", "mean"), viol=("viol", "mean"), copy=("copy_flag", "mean"),
             prompt_tok=("prompt_tokens", "mean"), tok_mean=("n_tokens", "mean"),
             tok_p50=("n_tokens", "median"), tok_p95=("n_tokens", lambda s: s.quantile(0.95)),
             lines_p50=("n_lines", "median"), commas_p50=("n_commas", "median"))
summ["cons_pass_stop"] = cal[~cal["trunc"]].groupby(["cons_id", "mode"])["cons_ok"].mean()
summ = summ.round(3)
summ.to_csv(f"{OUT_DIR}/calibration_v2_summary_tier{TIER}.csv")
display(summ)

pv = summ["cons_pass_stop"].unstack("mode")
bo = summ["base_ok"].unstack("mode")
tr = summ["trunc"].unstack("mode")
cp = summ["copy"].unstack("mode")
vi = summ["viol"].unstack("mode")

def classify(c):
    if bo.loc[c].min() < BASE_MIN:
        return "结构问题"
    if pv.loc[c].min() < LO:
        return "地板"
    if pv.loc[c].min() > HI:
        return "天花板"
    return "可用"

verdict = pd.DataFrame({"类别": {c: classify(c) for c in CANDIDATES},
                        "pass_DA": pv["DA"], "pass_CoT": pv["CoT"],
                        "base_DA": bo["DA"], "base_CoT": bo["CoT"],
                        "trunc_max": tr.max(axis=1), "copy_max": cp.max(axis=1), "DA_viol": vi["DA"]}).loc[CANDIDATES]
display(verdict)
SELECTED = [c for c in CANDIDATES if verdict.loc[c, "类别"] == "可用" and verdict.loc[c, "trunc_max"] < TRUNC_MAX
            and verdict.loc[c, "copy_max"] <= COPY_MAX and verdict.loc[c, "DA_viol"] <= DA_VIOL_MAX]
print("入选：", SELECTED)
print("其余：", {c: verdict.loc[c, "类别"] for c in CANDIDATES if c not in SELECTED})

print("\n按 d 的约束通过率（含 base_ok）：")
display(cal.pivot_table(index=["cons_id", "mode"], columns="d", values="cons_ok").round(2))
print("\n基线 F0：答案正确率 / token / 行数（按 d）：")
display(cal[cal.cons_id == "F0"].pivot_table(index="mode", columns="d", values=["answer_ok", "n_tokens", "n_lines"]).round(2))
print("\n约束对 CoT 准确率的影响（相对 F0）：")
display((summ["answer"].unstack("mode")["CoT"] - summ.loc["F0", "answer"]["CoT"]).round(3).rename("CoT 答案正确率 - F0"))

n  scope  cons_pass  cons_only  base_ok  answer  trunc  viol  \
cons_id mode                                                                   
F0      CoT   100   base       1.00       1.00     1.00    0.97   0.00  0.00   
        DA    100   base       1.00       1.00     1.00    0.29   0.00  0.09   
F2      CoT   100  final       0.96       0.96     1.00    0.96   0.00  0.00   
        DA    100  final       1.00       1.00     1.00    0.25   0.00  0.14   
F4      CoT   100  final       0.34       0.34     0.34    0.90   0.00  0.00   
        DA    100  final       1.00       1.00     1.00    0.28   0.00  0.00   
K1      CoT   100  final       0.06       0.06     0.99    0.63   0.01  0.00   
        DA    100  final       0.77       0.77     1.00    0.26   0.00  0.00   
K2      CoT   100  final       0.46       0.50     0.95    0.72   0.00  0.00   
        DA    100  final       0.31       0.31     1.00    0.27   0.00  0.00   
K4      CoT   100  final       0.47       0.52     0.90    0.73   0.00  0.00   
        DA    100  final       0.57       0.57     1.00    0.26   0.00  0.00   
L3      CoT   100   full       0.90       0.90     1.00    0.96   0.00  0.00   
        DA    100   full       0.93       0.93     0.93    0.67   0.07  1.00   
W1      CoT   100   full       0.27       0.27     1.00    0.93   0.00  0.00   
        DA    100   full       1.00       1.00     1.00    0.25   0.00  0.00   
W2      CoT   100   full       0.40       0.40     1.00    0.89   0.00  0.00   
        DA    100   full       1.00       1.00     1.00    0.33   0.00  0.36   
W3      CoT   100   full       0.01       0.01     1.00    0.88   0.00  0.00   
        DA    100   full       0.83       0.83     1.00    0.37   0.00  1.00   

              copy  prompt_tok  tok_mean  tok_p50  tok_p95  lines_p50  \
cons_id mode                                                            
F0      CoT    0.0      208.49    415.03    395.5   795.40       35.0   
        DA     0.0      217.49     10.67      8.0    23.10        1.0   
F2      CoT    0.0      227.49    413.31    396.0   801.00       35.0   
        DA     0.0      236.49     14.41      9.0    26.65        1.0   
F4      CoT    0.0      243.49    410.72    403.0   812.00       26.0   
        DA     0.0      252.49     12.24     12.0    13.00        2.0   
K1      CoT    0.0      244.49    421.93    403.0   803.00       26.0   
        DA     0.0      253.49     13.92     14.0    15.05        2.0   
K2      CoT    0.0      262.49    418.18    410.5   809.70       26.0   
        DA     0.0      271.49     13.29     13.0    14.00        2.0   
K4      CoT    0.0      243.49    412.46    393.0   800.10       26.0   
        DA     0.0      252.49     13.44     13.0    15.00        2.0   
L3      CoT    0.0      225.29    404.28    378.5   806.10       25.0   
        DA     0.0      234.29    201.30    170.0   512.00       14.0   
W1      CoT    0.0      232.49    346.57    339.0   785.30       20.0   
        DA     0.0      241.49      8.36      8.0    10.00        1.0   
W2      CoT    0.0      234.49    365.16    329.5   782.65       25.0   
        DA     0.0      243.49     21.68      8.5    79.20        1.0   
W3      CoT    0.0      231.49    350.72    334.5   756.25       20.0   
        DA     0.0      240.49     70.13     59.5   137.50       11.0   

              commas_p50  cons_pass_stop  
cons_id mode                              
F0      CoT          1.0           1.000  
        DA           0.0           1.000  
F2      CoT          1.0           0.960  
        DA           0.0           1.000  
F4      CoT          1.0           0.340  
        DA           0.0           1.000  
K1      CoT          1.0           0.061  
        DA           0.0           0.770  
K2      CoT          1.0           0.460  
        DA           0.0           0.310  
K4      CoT          1.0           0.470  
        DA           0.0           0.570  
L3      CoT          1.0           0.900  
   

,类别,pass_DA,pass_CoT,base_DA,base_CoT,trunc_max,copy_max,DA_viol
F2,天花板,1.00,0.960,1.00,1.00,0.00,0.0,0.14
F4,结构问题,1.00,0.340,1.00,0.34,0.00,0.0,0.00
K1,地板,0.77,0.061,1.00,0.99,0.01,0.0,0.00
K2,可用,0.31,0.460,1.00,0.95,0.00,0.0,0.00
K4,可用,0.57,0.470,1.00,0.90,0.00,0.0,0.00
W1,地板,1.00,0.270,1.00,1.00,0.00,0.0,0.00
W2,可用,1.00,0.400,1.00,1.00,0.00,0.0,0.36
W3,地板,0.83,0.010,1.00,1.00,0.00,0.0,1.00
L3,天花板,1.00,0.900,0.93,1.00,0.07,0.0,1.00


入选： ['K2', 'K4']
其余： {'F2': '天花板', 'F4': '结构问题', 'K1': '地板', 'W1': '地板', 'W2': '可用', 'W3': '地板', 'L3': '天花板'}

按 d 的约束通过率（含 base_ok）：


d               1     5     10    15    20
cons_id mode                              
F0      CoT   1.00  1.00  1.00  1.00  1.00
        DA    1.00  1.00  1.00  1.00  1.00
F2      CoT   1.00  1.00  0.90  0.95  0.95
        DA    1.00  1.00  1.00  1.00  1.00
F4      CoT   0.40  0.45  0.40  0.35  0.10
        DA    1.00  1.00  1.00  1.00  1.00
K1      CoT   0.00  0.00  0.05  0.20  0.05
        DA    0.20  0.95  1.00  1.00  0.70
K2      CoT   0.95  0.50  0.35  0.20  0.30
        DA    0.60  0.50  0.25  0.05  0.15
K4      CoT   1.00  0.60  0.20  0.40  0.15
        DA    1.00  1.00  0.35  0.30  0.20
L3      CoT   1.00  1.00  0.80  0.90  0.80
        DA    1.00  1.00  1.00  0.85  0.80
W1      CoT   0.25  0.20  0.20  0.25  0.45
        DA    1.00  1.00  1.00  1.00  1.00
W2      CoT   0.75  0.05  0.35  0.40  0.45
        DA    1.00  1.00  1.00  1.00  1.00
W3      CoT   0.00  0.00  0.05  0.00  0.00
        DA    0.35  0.90  1.00  0.95  0.95


基线 F0：答案正确率 / token / 行数（按 d）：


answer_ok                        n_lines                              \
d           1     5     10   15    20      1      5      10     15     20   
mode                                                                        
CoT        1.0  1.00  1.00  1.0  0.85    8.25  15.55  31.25  43.65  54.45   
DA         0.9  0.25  0.15  0.1  0.05    1.50   1.65   1.00   1.00   1.00   

     n_tokens                               
d          1      5       10     15     20  
mode                                        
CoT     88.90  192.6  401.55  607.6  784.5  
DA      14.55   13.9    8.30    8.3    8.3


约束对 CoT 准确率的影响（相对 F0）：


,CoT 答案正确率 - F0
cons_id,
F0,0.00
F2,-0.01
F4,-0.07
K1,-0.34
K2,-0.25
K4,-0.24
L3,-0.01
W1,-0.04
W2,-0.08


## 9. k 轴冒烟：入选约束累积叠加

按 `SELECTED` 顺序做 `k=1..K` 累积，小样本（`d ∈ {1,5,10}`，每档 10 题）。
观察 all-pass 是否**平滑下降**，而不是在某个 k 突然归零。入选少于 2 条时，手动改 `ORDER`。

In [11]:
ORDER = SELECTED if len(SELECTED) >= 2 else [c for c in CANDIDATES if c in ("F4", "K1", "W1", "W3")]
print("k 轴顺序：", ORDER)
kq = [q for q in QUESTIONS if q["d"] in (1, 5, 10) and q["i"] < 10]
run_jobs(make_jobs("kcurve", kq, ["DA", "CoT"], [ORDER[:k] for k in range(1, len(ORDER) + 1)]))
kc = load_results(stage="kcurve")
print(kc.groupby(["mode", "k"]).agg(n=("id", "size"), cons_all=("cons_ok", "mean"), answer=("answer_ok", "mean"),
                                   trunc=("trunc", "mean"), viol=("viol", "mean"),
                                   tok_mean=("n_tokens", "mean")).round(3))
pc = pd.DataFrame(kc["per_cons"].tolist()).add_prefix("pass_")
print(pd.concat([kc[["mode", "k"]], pc], axis=1).groupby(["mode", "k"]).mean().round(2))

k 轴顺序： ['K2', 'K4']
总计 120，已完成 0，待跑 120
[DA] 60/60  batch 5.7s
[CoT] 60/60  batch 63.6s
         n  cons_all  answer  trunc  viol  tok_mean
mode k                                             
CoT  1  30     0.700   0.633    0.0   0.0   233.000
     2  30     0.600   0.667    0.0   0.0   238.900
DA   1  30     0.333   0.233    0.0   0.0    13.267
     2  30     0.200   0.200    0.0   0.0    18.400
        pass_K2   pass_K4
mode k                   
CoT  1     0.77       NaN
     2     0.70       0.8
DA   1     0.33       NaN
     2     0.20  0.833333


## 10. 跑完之后要回填的东西

- Colab T4 上 4B 的实际 vLLM 参数 -> `AGENTS.md` 3.3 与 `record.md`
- `da_recheck` 的结论（是否采用加强版 DA preamble）-> `AGENTS.md` 4.1
- `verdict` 表与 `SELECTED` -> `record.md`，注明：入选规则是看了 v1 数据后的事后修订
- L3 的 `3d+4` 是否合适（看 CoT 的 `cons_pass` 按 d 的分布）
- 是否采纳 `scope="full"` 约束 -> 决定后再改 AGENTS 4.2